In [ ]:

import sys
from pathlib import Path

# Add the package source directory to path so aa_si_calibration is importable
# (not needed if the package is installed via pip install -e .)
sys.path.insert(0, str(Path("../src").resolve()))

from aa_si_calibration.calibration import generate_standardized_cal_mapping
from aa_si_calibration.mapping_algorithm import (
    get_calibration,
    get_calibration_from_file,
)

%load_ext autoreload
%autoreload 2

import echopype as ep
from pathlib import Path
import numpy as np

from aa_si_calibration import calibration
from aa_si_calibration import standardized_file_lib
from aa_si_calibration import manufacturer_file_parsers
from aa_si_calibration import comparison


from aa_si_ml import ml
from aa_si_utils import utils
from aa_si_visualization import assorted

import pyvista as pv

import xarray as xr


print("All imports successful!")


In [ ]:
# =============================================================================
# CONFIGURATION - Define Input and Output Paths
# =============================================================================


# If True, unused/rejected calibration files are moved to an "unused" folder
# instead of being permanently deleted.
KEEP_UNUSED_STANDARDIZED_FILES = True

# Record author - the individual running this pipeline and generating the records
RECORD_AUTHOR = "Placeholder Author"

# Cruise identifier for the calibration records
CRUISE_ID = "Pipeline_Example"

# Filename style for single-channel calibration files:
#   False -> long filenames derived from the full calibration key
#   True  -> short filenames: <date>_<freq_hz>_config-<N>.yml
SHORT_FILENAMES = True

# Conflict resolution strategy when a raw channel matches multiple cal files:
#   "interactive" -> prompt the user to choose which file to keep
#   "error"       -> raise an error listing the conflicts; the user must
#                     manually delete the unwanted file(s) and re-run the cell
CONFLICT_RESOLUTION = "error"

# Input folders
RAW_INPUT_FOLDER = Path("./raw_file_inputs")
# RAW_INPUT_FOLDER = Path("./example_data/ek80_CW_raw_file_input_folder")

CAL_INPUT_FOLDER = Path("../calibration_files/HB201607_cal")
# CAL_INPUT_FOLDER = Path("./example_data/ek80_cal_file_input_folder")

# Output folder (subdirectories are created automatically by the pipeline)
OUTPUT_BASE = Path("./nefsc_uc_e2_cal_outputs")

# Subdirectory paths (for use in step 4 examples)
SINGLE_CAL_OUTPUT = OUTPUT_BASE / "single_channel_calibration_files"

# Global calibration parameters applied to all single-channel files
global_params = {
    "cruise_id": CRUISE_ID,
    "record_author": RECORD_AUTHOR,
}

line_name = "sw_dive_profile"
line_file_path = '../line_files/SpermWhaleClicks_click_data_HB1603_SpermWhaleDive_Span0.2_07252016_2120_UTC.csv'

# User defined variables
raw_file_names = [] # Only use if populated with a list strings of raw file names
netcdf_output_folder_string = './NetCDF-files'
sv_output_folder_string = './Sv-files'
output_logs_folder_string = './Output-Logs'
clear_previous_json_logs = True

sv_flag_thresholds = {
                "critical_median": 2.0,
                "large_median": 1.0,
                "moderate_median": 0.5,
                "critical_max": 4.0,
                "large_max": 2.0,
                "moderate_max": 1.0
            }

print(f"Record author: {RECORD_AUTHOR}")
print(f"Cruise ID: {CRUISE_ID}")
print(f"Short filenames: {SHORT_FILENAMES}")
print(f"Keep unused standardized files: {KEEP_UNUSED_STANDARDIZED_FILES}")
print(f"Conflict resolution mode: {CONFLICT_RESOLUTION}")
print(f"\nInput folders:")
print(f"  Raw files:        {RAW_INPUT_FOLDER}")
print(f"  Calibration files: {CAL_INPUT_FOLDER}")
print(f"\nOutput base: {OUTPUT_BASE}")



In [ ]:

setup_result = utils.initial_setup_and_validation(
    raw_input_folder=RAW_INPUT_FOLDER,
    netcdf_output_folder_string=netcdf_output_folder_string,
    sv_output_folder_string=sv_output_folder_string,
    output_logs_folder_string=output_logs_folder_string,
    raw_file_names=raw_file_names,
    clear_previous_json_logs=clear_previous_json_logs,
)

netcdf_output_folder = setup_result["netcdf_output_folder"]
netcdf_output_path = setup_result["netcdf_output_path"]
sv_output_folder = setup_result["sv_output_folder"]
output_logs_folder = setup_result["output_logs_folder"]
raw_file_paths = setup_result["raw_file_paths"]


In [ ]:
# =============================================================================
# STEPS 1-3 + VERIFICATION: Full calibration pipeline
# =============================================================================
# 1. Read raw file configurations and save them
# 2. Read manufacturer calibration files and save to single-channel standardized files
# 3. Load single-channel files, match raw channels to calibration data, save mapping
# 4. Verify required parameters and file usage

pipeline_result = generate_standardized_cal_mapping(
    raw_input_folder=RAW_INPUT_FOLDER,
    cal_input_folder=CAL_INPUT_FOLDER,
    output_base=OUTPUT_BASE,
    global_params=global_params,
    short_filenames=SHORT_FILENAMES,
    keep_unused=KEEP_UNUSED_STANDARDIZED_FILES,
    conflict_resolution=CONFLICT_RESOLUTION,
)

mapping_dict = pipeline_result["mapping_dict"]
calibration_dict = pipeline_result["calibration_dict"]

In [ ]:

echodata = utils.open_and_combine_raw_files(raw_file_paths, netcdf_output_folder, sonar_model="EK60", include_bot=True)

In [ ]:
# extract calibration parameters from original netcdf file
params = calibration.extract_netcdf_calibration_parameters(echodata, output_logs_folder)

original_cal_params = params["cal_params"]
original_env_params = params["env_params"]
original_other_params = params["other_params"]

# the calibration parameters come from the first file in the sequential raw files that were combined by echopype
nc_cal_files = [str(raw_file_paths[0].name)]

original_other_params["source_filenames_across_channels"] = nc_cal_files
original_other_params["source_file_type"] = ".raw"

# print parameters from netcdf file
calibration.print_calibration_values(echodata, original_env_params, original_cal_params, original_other_params, "Calibration Values From .nc File")

In [ ]:
# =============================================================================
# Extract standardized calibration parameters in comparison format
# =============================================================================
# Convert the per-channel standardized data from calibration_dict into the
# (cal_params, env_params, other_params) structure needed by
# comparison.run_full_calibration_comparison.

standardized_params = calibration.extract_standardized_calibration_parameters(
    calibration_dict, mapping_dict, echodata=echodata,
)

standardized_cal_params = standardized_params["cal_params"]
standardized_env_params = standardized_params["env_params"]
standardized_other_params = standardized_params["other_params"]

standardized_other_params["source_filenames_across_channels"] = [str(raw_file_paths[0].name)]
standardized_other_params["source_file_type"] = ".cal"

calibration.print_calibration_values(echodata, standardized_env_params, standardized_cal_params, standardized_other_params, "Calibration Values From Standardized Files")

In [ ]:


comparison_results = comparison.run_full_calibration_comparison(
    echodata,
    standardized_cal_params,
    standardized_env_params,
    standardized_other_params,
    original_cal_params,
    original_env_params,
    original_other_params,
    output_logs_folder,
    sv_output_folder,
    sv_flag_thresholds=sv_flag_thresholds,
    mask_seafloor_buffer_m=100
)

ds_Sv_baseline = comparison_results["ds_Sv_baseline"]
ds_Sv_calibrated = comparison_results["ds_Sv_combined_test"]
mask = comparison_results["mask"]
verification_results = comparison_results["verification_results"]

In [ ]:
ds_Sv_clean, ds_MVBS = ml.data_preprocessing_pipeline(
    ds_Sv_baseline,
    echodata,
    noise_range_sample_num=10,
    noise_ping_num=5,
    mvbs_range_bin="2m",
    mvbs_ping_time_bin="10s",
    mvbs_nan_threshold=.6,
    plot_window=[0, 1800, 0, 515],
    overlay_line_var=line_name,
    overlay_line_path=line_file_path
    )

In [ ]:
cluster_colors = [
                "#FFAD09", "#35E200", "#FF0000", "#2F00FF", "#DC0303",
                "#FFFB1C", "#4E9200", "#970021", "#8E00E0", "#017685FF", "#4100B9FF"
            ]
dataset_name = "ml_dataset_6"
custom_normalization_name = "normalized_data_flatten_mean_centered_6"
ml_result_name = "dbscan_background_results_6"

ds_MVBS, gridded_background_results_dbscan, dbscan_results = ml.full_dbscan_iteration(
    ds_MVBS,
    dataset_name,
    ds_Sv_clean,
    feature_strategy="mean_centered", # CLR
    data_var="Sv",
    custom_normalization_name=custom_normalization_name,
    ml_result_name=ml_result_name,
    normalization_strategy="flatten", # Yeo Johnson + PIT
    feature_weights=[1, 1, 1],
    plot_window=[0, 1800, 0, 515],
    min_samples=10, # usually ~ 2-10
    sample_size=1000000, # use full dataset if smaller
    min_cluster_size=1400, # usually ~ 1000 - 2000 for larger datasets
    cluster_selection_method="eom", # also worth trying leaf
    useHDBScan=True,
    baseline_channel=0,
    # soft_membership_threshold=.6,
    cluster_colors=cluster_colors,
    overlay_line_var=line_name
    )
